In [31]:
import pandas as pd
import re
import nltk
nltk.download('stopwords')
data = pd.read_csv("spamham.csv")
data.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Label,Message
0,ham,Subject: enron methanol ; meter # : 988291\r\n...
1,ham,"Subject: hpl nom for january 9 , 2001\r\n( see..."
2,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar..."
3,spam,"Subject: photoshop , windows , office . cheap ..."
4,ham,Subject: re : indian springs\r\nthis deal is t...


In [32]:
print(data.columns.tolist())

['Label', 'Message']


In [33]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import re

ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
corpus = []

for i in range(len(data)):
    review = re.sub('[^a-zA-Z]', ' ', data['Message'][i])
    review = review.lower().split()
    review = [ps.stem(word) for word in review if word not in stop_words]
    corpus.append(' '.join(review))

print("Corpus built successfully. Sample:", corpus[0])

Corpus built successfully. Sample: subject enron methanol meter follow note gave monday preliminari flow data provid daren pleas overrid pop daili volum present zero reflect daili activ obtain ga control chang need asap econom purpos


In [34]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
x = cv.fit_transform(corpus)

In [35]:
x

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 343391 stored elements and shape (10162, 40272)>

In [36]:
y = data['Label'].map({'ham':0, 'spam':1})

In [37]:
y

0        0
1        0
2        0
3        1
4        0
        ..
10157    1
10158    0
10159    0
10160    0
10161    0
Name: Label, Length: 10162, dtype: int64

## Grid Search

- Why do we use Grid Search?

`GridSearchCV` is a technique to search through the best parameter values from the given set of the grid of parameters. It is basically a cross-validation method. the model and the parameters are required to be fed in. Best parameter values are extracted and then the predictions are made.

## Select the best model
- so here we have some list of the best text classification algorithms we imported. Now we will compare each model's score and see which model is performing better than rest of the others

### 1. Multinomial Naive Bayes Classifier

The multinomial NB classifier has a hyperparameter called **`alpha`**. It is the **smoothing parameter** to avoid **zero counts** when calculating the frequencies. 

For example, if we are now classifying a new SMS with a word "ryan" which never exist in the spam emails within our training dataset, the **likelihood** for this word will be zero. This will casue the **overall likelihood** to be zero (because we take the product of all **individual likelihoods**) for no matter what class of output variable we have.

Therefore, we need to add **additional counts** to each word when calculating the frequencies to avoid have a zero likelihood value. **Alpha** indicates how many **additional counts** we add.

### 2. Gaussian Naive Bayes Classifier

There is one hyperparameter we need to tune: **`var_smoothing`**. This is the **portion of the largest variance** of all features that is added to variances for **calculation stability**.

### 3. SVC
SVC, or Support Vector Classifier, is a supervised machine learning algorithm typically used for classification tasks. SVC works by mapping data points to a high-dimensional space and then finding the optimal hyperplane that divides the data into two classes.

In [38]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Gaussian Naive Bayes": GaussianNB(),
    "SVC": SVC()
}

- ### We will create a generic function to check each model's performance so that we can compare those

In [39]:
# Create a function which can evaluate models and return a report 

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_models(X, y, models):
    '''
    This function takes in X and y and models dictionary as input
    It splits the data into Train Test split
    Iterates through the given model dictionary and evaluates the metrics
    Returns: Dataframe which contains report of all models metrics with cost
    '''
    # separate dataset into train and test
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
    

    models_list = []
    scores = []
    
    for i in range(len(list(models))):
        model = list(models.values())[i]
        model.fit(X_train, y_train) # Train model

        # Make predictions
        y_pred = model.predict(X_test)

        score = accuracy_score(y_test,y_pred)
        
        model_name = list(models.keys())[i]
        print(f'---- score for --- {model_name} ----')
        print(f"{score}")
        models_list.append(model_name)
        scores.append(score)
    
    print()
    
    report = pd.DataFrame()
    report['Model_name'] = models_list
    report['Score'] = scores        
    return report

In [51]:
X = tfidf.fit_transform(corpus).toarray()
y = pd.get_dummies(data['label'], drop_first=True).values.ravel()

print("✅ X shape:", X.shape)
print("✅ y shape:", y.shape)

✅ X shape: (10162, 3000)
✅ y shape: (10162,)


In [52]:
import re
import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('stopwords')

# ── Step 1: Fix columns ──────────────────
data = data.rename(columns={'Label': 'label', 'Message': 'message'})

# ── Step 2: Preprocess text ──────────────
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))
corpus = []

for i in range(len(data)):
    review = re.sub('[^a-zA-Z]', ' ', data['message'][i])
    review = review.lower().split()
    review = [ps.stem(word) for word in review if word not in stop_words]
    corpus.append(' '.join(review))

print(f"✅ Corpus ready — {len(corpus)} messages")

# ── Step 3: TF-IDF ───────────────────────
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(corpus).toarray()
y = pd.get_dummies(data['label'], drop_first=True).values.ravel()

print(f"✅ X shape: {X.shape}")
print(f"✅ y shape: {y.shape}")

# ── Step 4: Run evaluate_models ──────────
report = evaluate_models(X, y, models)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\himan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


✅ Corpus ready — 10162 messages
✅ X shape: (10162, 3000)
✅ y shape: (10162,)
---- score for --- Multinomial Naive Bayes ----
0.9478603049680275
---- score for --- Gaussian Naive Bayes ----
0.7629119527791441
---- score for --- SVC ----
0.9670437776684703



In [53]:
report = evaluate_models(X, y, models)

---- score for --- Multinomial Naive Bayes ----
0.9478603049680275
---- score for --- Gaussian Naive Bayes ----
0.7629119527791441
---- score for --- SVC ----
0.9670437776684703



In [54]:
report.sort_values('Score')

,Model_name,Score
1,Gaussian Naive Bayes,0.762912
0,Multinomial Naive Bayes,0.947860
2,SVC,0.967044


- ### From the report above we can see that the Gaussian Naive Bayes model performed the best, so we will continue training our model using Gaussian Naive Bayes algorithm.

In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [56]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV

params = {
    'alpha': [0.1, 0.5, 1.0, 2.0]
}

mnb = MultinomialNB()
mnb_cv = GridSearchCV(mnb, params, cv=10)

mnb_cv.fit(X_train, y_train)

print("Best Parameters:", mnb_cv.best_params_)
print("Best Accuracy:", mnb_cv.best_score_)


Best Parameters: {'alpha': 2.0}
Best Accuracy: 0.9463660104581342


In [58]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

# ── Step 1: Train/Test Split ─────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Step 2: Find Best Params ─────────────
params = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]}
gnb_cv = GridSearchCV(GaussianNB(), params, cv=5, scoring='accuracy')
gnb_cv.fit(X_train, y_train)
print("✅ Best params:", gnb_cv.best_params_)

# ── Step 3: Train Final Model ────────────
spam_detect_model = GaussianNB(**gnb_cv.best_params_)
spam_detect_model.fit(X_train, y_train)

# ── Step 4: Predict & Evaluate ───────────
y_pred = spam_detect_model.predict(X_test)
confusion_m = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Accuracy of the model is {accuracy * 100:.2f}%")
print(f"\n📊 Confusion Matrix:\n{confusion_m}")
print(f"\n        Predicted Ham  Predicted Spam")
print(f"Actual Ham     {confusion_m[0][0]}           {confusion_m[0][1]}")
print(f"Actual Spam    {confusion_m[1][0]}           {confusion_m[1][1]}")

✅ Best params: {'var_smoothing': 1e-05}

✅ Accuracy of the model is 85.83%

📊 Confusion Matrix:
[[1360  247]
 [  41  385]]

        Predicted Ham  Predicted Spam
Actual Ham     1360           247
Actual Spam    41           385


- So we can see that the model performed well and we have got an accuracy of 96% which is pretty insane. In our project we will be having all these models and we will be selecting the models based on the performence.

In [59]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score

# MultinomialNB works much better with TF-IDF
model = MultinomialNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
confusion_m = confusion_matrix(y_test, y_pred)

print(f"✅ Accuracy: {accuracy * 100:.2f}%")
print(f"\n📊 Confusion Matrix:\n{confusion_m}")

✅ Accuracy: 94.79%

📊 Confusion Matrix:
[[1569   38]
 [  68  358]]


In [61]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "SVM"                 : SVC(kernel='linear')
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cm  = confusion_matrix(y_test, y_pred)
    print(f"\n✅ {name} Accuracy: {acc * 100:.2f}%")
    print(f"📊 Confusion Matrix:\n{cm}")


✅ Logistic Regression Accuracy: 94.93%
📊 Confusion Matrix:
[[1596   11]
 [  92  334]]

✅ SVM Accuracy: 97.20%
📊 Confusion Matrix:
[[1590   17]
 [  40  386]]


In [63]:
import pickle
import re

# Save SVM model as variable first
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)
print("✅ SVM model trained and saved as variable")

# Save model and tfidf vectorizer
with open('spam_detector_model.pkl', 'wb') as f:
    pickle.dump(svm_model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("✅ Model and vectorizer saved to disk!")

# Test prediction
def predict_spam(message):
    review = re.sub('[^a-zA-Z]', ' ', message).lower().split()
    review = [ps.stem(word) for word in review if word not in stop_words]
    vector = tfidf.transform([' '.join(review)]).toarray()
    result = svm_model.predict(vector)[0]
    return "🚨 SPAM" if result == 1 else "✅ HAM (Not Spam)"

# Test it
print("\n🔍 Test Predictions:")
print(predict_spam("Congratulations! You won a free prize. Click here now!"))
print(predict_spam("Hey, are we still meeting for lunch tomorrow?"))

✅ SVM model trained and saved as variable
✅ Model and vectorizer saved to disk!

🔍 Test Predictions:
🚨 SPAM
✅ HAM (Not Spam)
